# 5 · MCP (Model Context Protocol) — Servidor y cliente mínimos
### Sesión 1 — Agentes de IA, Orquestación y Protocolos

**Complejidad: 🟠 Alta**   |   **Dependencias: `mcp`, `anthropic`, `nest_asyncio`**

Este notebook usa código asíncrono (el protocolo MCP corre sobre `asyncio`) y lanza un **subproceso** (el servidor MCP corre aparte del cliente). Si tuviste problemas con notebooks anteriores por instalación, este es independiente de LangChain y LlamaIndex — solo necesita el SDK de `mcp`.


## 🎯 Objetivo de aprendizaje

Al terminar este notebook vas a poder:
- Explicar el problema de integraciones "a la medida" que MCP resuelve (N modelos × M herramientas).
- Describir la arquitectura cliente-servidor de MCP y sus 3 primitivos: Tools, Resources y Prompts.
- Levantar un servidor MCP mínimo, descubrir sus herramientas desde un cliente, e invocarlas dentro de un agente.


## 📚 Teoría: MCP (Model Context Protocol)

Antes de MCP, conectar un modelo con una herramienta externa (GitHub, una base de datos, Slack) requería una integración hecha a la medida para cada combinación de modelo y herramienta — si tenías N modelos y M herramientas, terminabas con N×M integraciones distintas que mantener.

**MCP** estandariza esa conexión con una arquitectura **cliente-servidor**:
- El **servidor MCP** expone capacidades de un sistema externo mediante 3 primitivos: **Tools** (acciones que el modelo puede ejecutar), **Resources** (datos de contexto que puede leer) y **Prompts** (plantillas reutilizables).
- El **cliente MCP** (la app de IA, un IDE, un agente) se conecta a ese servidor y **descubre automáticamente** qué capacidades ofrece, sin tener que conocerlas de antemano en su código.

El flujo de comunicación es: **handshake** (negociar la conexión) → **descubrimiento** (`list_tools()`) → **invocación** → **respuesta** → **integración** al contexto del modelo.

**La diferencia clave frente al function calling del notebook 1:** ahí, las herramientas estaban *hardcodeadas* en tu código. Con MCP, si el servidor agrega una herramienta nueva, cualquier cliente MCP la descubre automáticamente — sin cambiar una línea de su propio código. MCP es, en el fondo, function calling estandarizado y portable entre distintas aplicaciones y modelos.


## 0. Instalación

In [ ]:
!pip install -q "mcp[cli]" anthropic nest_asyncio

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # necesario para correr código async de MCP dentro de un notebook


### Configurar API key de Anthropic

**Cómo obtenerla:** [console.anthropic.com](https://console.anthropic.com/settings/keys)

Recomendado en Colab: guárdala en **Secrets** (ícono de llave 🔑 a la izquierda) con el nombre `ANTHROPIC_API_KEY` y actívala para este notebook. Si no usas Secrets, te la pedirá por input.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = os.environ.get("ANTHROPIC_API_KEY") or getpass("Pega tu ANTHROPIC_API_KEY: ")

print("API key configurada:", "OK" if os.environ.get("ANTHROPIC_API_KEY") else "FALTA")


## 1. Escribir un servidor MCP

Un servidor MCP expone herramientas (`tools`) que cualquier cliente MCP puede descubrir e invocar. Lo escribimos como un archivo `.py` aparte: en un entorno real este servidor corre como un proceso independiente.


In [ ]:
servidor_mcp_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("universidad-server")

@mcp.tool()
def consultar_horario(materia: str) -> str:
    """Consulta el horario de una materia de la universidad."""
    horarios = {
        "algoritmos": "Lunes y miércoles, 8:00am - 10:00am, Salón 204",
        "bases de datos": "Martes y jueves, 2:00pm - 4:00pm, Lab 3",
    }
    for k, v in horarios.items():
        if k in materia.lower():
            return v
    return f"No se encontró horario para \'{materia}\'"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("servidor_universidad_mcp.py", "w") as f:
    f.write(servidor_mcp_code)

print("Servidor MCP escrito en servidor_universidad_mcp.py")


## 2. Cliente MCP: handshake, descubrimiento e invocación

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import asyncio

async def probar_mcp():
    server_params = StdioServerParameters(command="python3", args=["servidor_universidad_mcp.py"])

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # --- Handshake ---
            await session.initialize()

            # --- Descubrimiento: ¿qué tools expone el servidor? ---
            tools_response = await session.list_tools()
            print("Herramientas descubiertas automáticamente:")
            for t in tools_response.tools:
                print(f"  - {t.name}: {t.description}")

            # --- Invocación ---
            result = await session.call_tool("consultar_horario", arguments={"materia": "algoritmos"})
            print("\nResultado de la invocación:")
            print(result.content[0].text)

await probar_mcp()


**Lo importante de este ejemplo:** el cliente **nunca tuvo que conocer de antemano** cuáles eran las herramientas del servidor — las descubrió en tiempo de ejecución con `list_tools()`. Si mañana agregas una nueva `@mcp.tool()` al servidor, el cliente la verá automáticamente sin cambiar una línea de su código.

## 🧪 Ejercicio

Agrega una segunda herramienta al servidor (`consultar_disponibilidad_sala`) y vuelve a correr el cliente. Confirma que aparece en `list_tools()` sin haber modificado el código del cliente.


In [ ]:
# Tu código aquí (edita servidor_mcp_code, vuelve a escribir el archivo y vuelve a correr probar_mcp())


## 3. Conectar el servidor MCP a un agente con tool use

Para que el LLM realmente use las herramientas del servidor MCP dentro de un agente, el cliente debe traducir el `list_tools()` de MCP al formato `tools=[...]` que espera la API de Anthropic.


In [ ]:
import anthropic
client = anthropic.Anthropic()

def mcp_tools_to_anthropic_format(mcp_tools):
    """Convierte la lista de tools MCP al formato que espera la API de Anthropic."""
    return [
        {
            "name": t.name,
            "description": t.description,
            "input_schema": t.inputSchema,
        }
        for t in mcp_tools
    ]

async def agente_con_mcp(pregunta: str):
    server_params = StdioServerParameters(command="python3", args=["servidor_universidad_mcp.py"])
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_tools = (await session.list_tools()).tools
            anthropic_tools = mcp_tools_to_anthropic_format(mcp_tools)

            messages = [{"role": "user", "content": pregunta}]
            response = client.messages.create(
                model="claude-sonnet-4-6", max_tokens=1024,
                tools=anthropic_tools, messages=messages,
            )

            if response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        result = await session.call_tool(block.name, arguments=block.input)
                        tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                              "content": result.content[0].text})
                messages.append({"role": "user", "content": tool_results})
                response = client.messages.create(model="claude-sonnet-4-6", max_tokens=1024,
                                                    tools=anthropic_tools, messages=messages)

            return "".join(b.text for b in response.content if b.type == "text")

respuesta = await agente_con_mcp("¿A qué hora es la clase de bases de datos?")
print(respuesta)


---
**Siguiente notebook:** `taller5_prototipo_agente.ipynb` — plantilla del taller (usa solo `anthropic`, reutilizando el patrón del notebook 2).
